In [2]:
!pip install -q -U transformers datasets accelerate bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [3]:
import torch, re, json, random
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

TID = "Qwen/Qwen2.5-7B-Instruct"
tok = AutoTokenizer.from_pretrained(TID)
tok.padding_side = "left"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
teacher = AutoModelForCausalLM.from_pretrained(TID, quantization_config=bnb, device_map={"": 0})

raw = load_dataset("HuggingFaceH4/no_robots", split="train")
pool = [r["prompt"] for r in raw
        if r["category"] in ("Open QA", "Closed QA", "Brainstorm")
        and 20 < len(r["prompt"]) < 300
        and "chatbot" not in r["prompt"].lower()]
random.seed(7); random.shuffle(pool)
print(len(pool), "available")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

... (8 outputs left out)

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

2087 available


In [4]:
SYSTEM = ("You are a careful, accurate assistant. Answer using EXACTLY this format and nothing else:\n"
          "Summary: <one sentence that directly answers the question>\n"
          "Key points:\n- <point>\n- <point>\n- <point>\n"
          "Next step: <one sentence of practical advice>\n"
          "Rules: Stay on topic. If the user asks for a list or several items, put the three best items "
          "in Key points. Only state facts you are confident about. If you are unsure, say so in the "
          "Summary instead of guessing. Never invent names, dates or numbers.")

def fix(a):
    a = a.replace("**", "")
    if re.search(r"[\u4e00-\u9fff\u3040-\u30ff]", a): return None
    lines = [l.rstrip() for l in a.splitlines() if l.strip()]
    if len(lines) != 6: return None
    if not lines[0].startswith("Summary:") or lines[1].strip() != "Key points:": return None
    if not lines[5].startswith("Next step:"): return None
    if len(lines[0]) > 320 or "<" in a: return None
    bullets = []
    for l in lines[2:5]:
        if not re.match(r"^\s*(?:[-*•]|\d+[.)])\s+\S", l): return None
        bullets.append("- " + re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", l))
    return "\n".join([lines[0], "Key points:", *bullets, lines[5]])

all_q = pool[:1100]
all_a = []
for i in range(0, len(all_q), 16):
    batch = all_q[i:i+16]
    texts = [tok.apply_chat_template(
                [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}],
                add_generation_prompt=True, tokenize=False) for q in batch]
    inp = tok(texts, return_tensors="pt", padding=True).to(teacher.device)
    with torch.no_grad():
        out = teacher.generate(**inp, max_new_tokens=220, do_sample=False,
                               repetition_penalty=1.05, pad_token_id=tok.eos_token_id)
    for o in out:
        all_a.append(tok.decode(o[inp["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    json.dump(list(zip(all_q, all_a)), open("/kaggle/working/v2_raw_partial.json", "w"))
    print(i, end=" ")

seen, clean = set(), []
for q, a in zip(all_q, all_a):
    f = fix(a)
    if f and q not in seen:
        seen.add(q)
        clean.append({"messages": [{"role": "user", "content": q},
                                   {"role": "assistant", "content": f}]})
random.seed(42); random.shuffle(clean)
n_val = max(40, len(clean) // 10)
val, train = clean[:n_val], clean[n_val:]
print("\n", len(clean), "clean |", len(train), "train |", len(val), "val")
json.dump(train, open("/kaggle/working/v2_train.json", "w"))
json.dump(val, open("/kaggle/working/v2_val.json", "w"))

0 16 32 48 64 80 96 112 128 144 160 176 192 208 224 240 256 272 288 304 320 336 352 368 384 400 416 432 448 464 480 496 512 528 544 560 576 592 608 624 640 656 672 688 704 720 736 752 768 784 800 816 832 848 864 880 896 912 928 944 960 976 992 1008 1024 1040 1056 1072 1088 
 792 clean | 713 train | 79 val
